# Wider-view test: current model against a longer IQ view

Trains two models on all 60 classes, with random frequency placement left on as it is in the shards. Both use the same windows, seed and settings. The only difference is `--wide-iq`.

This uses a quarter of every shard for 15 epochs, so it answers one question: **does the wider view help, and does it hurt any class?** Neither model is an ensemble member. Don't report their scores, and don't save them to `members/`.

Before you start: **Runtime → Change runtime type → GPU**. Expect about 5 to 6 hours for both runs.


## 1. Get the code


In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -b eileen-17class-pilot https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!pip install -q pyyaml h5py gdown


## 2. Check the GPU

This must say `CUDA: True`. If it doesn't, change the runtime type and start again.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Turn on the GPU: Runtime -> Change runtime type'


## 3. Save results to your Drive

Checkpoints go here every epoch, so a dropped session can carry on.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nexa_wide_iq_test'
!mkdir -p "$OUT/current" "$OUT/wide"


## 4. Download the shards

About 3.8 GB from `processed_data`. The next step checks every file against its manifest, so a bad download stops the run.


In [ ]:
FOLDERS = {
    'jam': '1hPb-Hi5-RNXHOPB7eqs5o7oSJ66LZ0Fu',
    'pulse': '18jZ_2gJ1HbCloRw4Au6KbPuUrhJyP3MW',
    'spread': '1xr31dYor-rUnXdyXpK512t-HeGmHyugz',
    'radar': '1rGQmzS3dz0-0GE6Ixk7cVkSUtX6daUel',
    'fskpsk': '15aB-sjz_hWMcejUo0JIwd0S2cackg_Y8',
    'analog': '1oz2amRn65zqJ_JWymViaPhZ0wANdFLuZ',
    'multicarrier': '16FsXoJ4yc--pvm3yv89vd8RhTPh3VR2o',
    'apsk': '1uidz9R9NUz91mYe9ko1o8gBzX5Z8sedr',
}
for group, folder in FOLDERS.items():
    !gdown --folder "https://drive.google.com/drive/folders/{folder}" -O "/content/shards/{group}"
!du -sh /content/shards/*


## 5. Train

Check first, a few minutes on a small sample. If this fails, stop and fix it.


In [ ]:
GROUPS = 'jam pulse spread radar fskpsk analog multicarrier apsk'
RUN = f'python scripts/pilot_train_shards.py --shards /content/shards --groups {GROUPS}'
!{RUN} --out /content/check --fraction 0.005 --epochs 1 --wide-iq


The current model. If the session drops, run cells 1 to 4 and the `GROUPS` cell again, then this one: it resumes from your Drive.


In [ ]:
!{RUN} --out "$OUT/current" --fraction 0.25 --epochs 15


The wider view. It resumes the same way.


In [ ]:
!{RUN} --out "$OUT/wide" --fraction 0.25 --epochs 15 --wide-iq


## 6. Compare

Biggest losses first, so anything the wider view hurts is at the top. Tell Claude when this is done and it can read both scorecards from your Drive.


In [ ]:
import json
cur = json.load(open(f'{OUT}/current/scorecard.json'))
wide = json.load(open(f'{OUT}/wide/scorecard.json'))
for key in ('macro_top1', 'macro_recall'):
    print(f'{key}: current {cur[key]:.3f}, wide {wide[key]:.3f}')
rows = [(wide['per_class'][n]['top1_recall'] - c['top1_recall'], n, c['top1_recall'], wide['per_class'][n]['top1_recall'])
        for n, c in cur['per_class'].items()]
for change, name, before, after in sorted(rows):
    print(f'{name:<16} top-1 {before:.3f} -> {after:.3f}  ({change:+.3f})')
